# Step 5 – Aggregate results (Tables 3–9, 13)

Averages the fold results, runs the Friedman test, computes Kuncheva stability, robust rank aggregation and the SHAP and permutation summaries, and writes `summary.pkl`.

*Part of the code accompanying* **Explainable, Leakage-Free Gene Selection for Breast Cancer Histologic Grade** (Lamba, Munjal, Gigras). Run the notebooks from the folder that holds them, with the exported CSV files in `./data`.

In [ ]:
import glob, pickle, itertools, numpy as np, pandas as pd
from scipy.stats import friedmanchisquare, kendalltau, beta, rankdata
%run pipeline.ipynb

C4 = ["GSE7390", "UPP", "MAINZ", "UNT"]
NG = 12688
A = {}

In [ ]:
def lk(a, b, n=NG):
    """Kuncheva consistency with Lustgarten's correction for unequal sizes."""
    ka, kb, r = len(a), len(b), len(a & b)
    den = min(ka, kb) - max(0, ka + kb - n)
    return (r - ka * kb / n) / den if den else 0.0

In [ ]:
def stability(sets):
    return float(np.mean([lk(a, b) for a, b in itertools.combinations(sets, 2)]))

In [ ]:
for c in C4:
    F = [pickle.load(open(f, "rb")) for f in sorted(glob.glob(f"folds_{c}/r*_k*.pkl"))]
    meta = pickle.load(open(f"folds_{c}/meta.pkl", "rb"))
    leaky = pickle.load(open(f"folds_{c}/leaky.pkl", "rb"))
    d = {"n": meta["n"], "classes": meta["classes"], "nfolds": len(F)}
    d["n_inf"] = np.mean([f["n_inf"] for f in F]); d["n_cfs"] = np.mean([f["n_cfs"] for f in F])
    d["n_cons"] = np.mean([f["n_cons"] for f in F])
    d["n_cfs_rng"] = (min(f["n_cfs"] for f in F), max(f["n_cfs"] for f in F))
    d["n_cons_rng"] = (min(f["n_cons"] for f in F), max(f["n_cons"] for f in F))
    tab = {}
    for cl in CLASSIFIERS:
        R = [f["res"][cl] for f in F]
        tab[cl] = {k: (np.mean([r[k] for r in R]), np.std([r[k] for r in R])) for k in
                   ["macro_f1", "mcc", "bal_acc", "macro_precision", "macro_recall", "fallout"]}
        if cl == "HWNB":
            for k in ["recall_g", "f1_g", "fallout_g"]:
                tab[cl][k] = np.mean([r[k] for r in R], 0)
    d["tab"] = tab
    d["leaky"] = {cl: {k: np.mean([r[k] for r in v]) for k in ["macro_f1", "macro_precision", "macro_recall", "fallout", "mcc"]}
                  for cl, v in leaky["res"].items()}
    d["leaky_genes"] = (leaky["n_cfs"], leaky["n_cons"], list(leaky["genes"]))
    sets = [set(f["genes"]) for f in F]
    d["stability"] = stability(sets)
    freq = pd.Series([g for s in sets for g in s]).value_counts() / len(F)
    d["freq"] = freq
    # ---- SHAP (rep 0: every tumour once). mean |phi| per gene and class over all explained tumours
    S = [f for f in F if "shap" in f]
    nexp = sum(len(f["te"]) for f in S)
    shap_sum = {}; shap_cond = {}
    for f in S:
        a = np.abs(f["shap"])  # n x m x 3
        for j, g in enumerate(f["genes"]):
            shap_sum[g] = shap_sum.get(g, 0) + a[:, j, :].sum(0)
            shap_cond.setdefault(g, []).append(a[:, j, :].mean(0))
    d["shap_mean"] = pd.DataFrame({g: v / nexp for g, v in shap_sum.items()}, index=["G1", "G2", "G3"]).T
    d["nexp"] = nexp
    # ---- permutation importance, conditional on selection, and over all folds
    perm = {}
    for f in F:
        for j, g in enumerate(f["genes"]):
            perm.setdefault(g, []).append((f["perm_grouped"][j], f["perm_single"][j]))
    d["perm"] = pd.DataFrame({g: [np.mean([p[0] for p in v]), np.mean([p[1] for p in v]), len(v)] for g, v in perm.items()},
                             index=["grouped", "single", "nsel"]).T
    d["perm"]["grouped_all"] = d["perm"]["grouped"] * d["perm"]["nsel"] / len(F)
    # ---- rank agreement over genes selected in >=10% of folds
    U = freq[freq >= 0.10].index
    sm = d["shap_mean"].reindex(U).fillna(0).sum(1)
    pg = d["perm"].reindex(U)["grouped_all"].fillna(0)
    fr = freq.reindex(U)
    d["agree_universe"] = len(U)
    d["tau"] = {"sel_vs_shap": kendalltau(fr, sm), "sel_vs_perm": kendalltau(fr, pg), "shap_vs_perm": kendalltau(sm, pg)}
    top = 5
    d["robust"] = sorted(set(fr.nlargest(top).index) & set(sm.nlargest(top).index) & set(pg.nlargest(top).index))
    d["rank_tables"] = pd.DataFrame({"freq": fr, "shap": sm, "perm": pg}).sort_values("freq", ascending=False)
    # ---- out-of-fold HWNB probabilities (average over repeats)
    P = np.zeros((meta["n"], 3)); cnt = np.zeros(meta["n"])
    for f in F:
        P[f["te"]] += f["proba_hwnb"]; cnt[f["te"]] += 1
    d["proba_oof"] = P / cnt[:, None]
    d["samples"] = meta["samples"]
    A[c] = d

### Friedman across cohorts on macro-F1

In [ ]:
# ---- Friedman across cohorts on macro-F1
M = np.array([[A[c]["tab"][cl]["macro_f1"][0] for cl in CLASSIFIERS] for c in C4])
fr_stat = friedmanchisquare(*M.T)
ranks = np.array([rankdata(-row) for row in M]).mean(0)
k, N = len(CLASSIFIERS), len(C4)
CD = 3.219 * np.sqrt(k * (k + 1) / (6 * N))   # Nemenyi q_0.05 for k = 11

### Robust rank aggregation (Kolde et al. 2012) across cohorts on selection-frequency ranks

In [ ]:
# ---- robust rank aggregation (Kolde et al. 2012) across cohorts on selection-frequency ranks
def rra(lists, universe):
    out = {}
    for g in universe:
        r = sorted([(lst.index(g) + 1) / len(universe) if g in lst else 1.0 for lst in lists])
        nL = len(r)
        p = min(beta.cdf(r[i], i + 1, nL - i) for i in range(nL))
        out[g] = min(1.0, p * nL)
    s = pd.Series(out).sort_values()
    m = len(s); q = (s * m / np.arange(1, m + 1))[::-1].cummin()[::-1].clip(upper=1)
    return pd.DataFrame({"rho": s, "q": q})


lists = [list(A[c]["freq"].index) for c in C4]
universe = sorted(set().union(*lists))
R = rra(lists, universe)
R["cohorts"] = [sum(g in A[c]["freq"].index for c in C4) for g in R.index]
R["mean_freq"] = [np.mean([A[c]["freq"].get(g, 0) for c in C4]) for g in R.index]

### Ablation

In [ ]:
# ---- ablation
abl = {}
for f in sorted(glob.glob("abl/*.pkl")):
    r = pickle.load(open(f, "rb"))
    for v, d in r.items():
        abl.setdefault(v, {"f1": [], "mcc": [], "sets": []})
        for kk in d: abl[v][kk] += d[kk]
ABL = {v: {"f1": (np.mean(d["f1"]), np.std(d["f1"])), "mcc": np.mean(d["mcc"]), "size": np.mean([len(s) for s in d["sets"]]),
           "stab": stability(d["sets"]), "n": len(d["f1"])} for v, d in abl.items()}
LODO = {c: pickle.load(open(f"lodo_{c}.pkl", "rb"))[c] for c in C4}
pickle.dump({"A": A, "fried": (fr_stat, ranks, CD), "RRA": R, "ABL": ABL, "LODO": LODO}, open("summary.pkl", "wb"))

### Print

In [ ]:
# ---------------- print
for c in C4:
    d = A[c]
    print(f"\n== {c} n={d['n']} classes={d['classes']} folds={d['nfolds']} inf={d['n_inf']:.0f} cfs={d['n_cfs']:.1f}{d['n_cfs_rng']} cons={d['n_cons']:.1f}{d['n_cons_rng']} stab={d['stability']:.3f}")
    for cl in CLASSIFIERS:
        t = d["tab"][cl]
        print(f"  {cl:5s} F1 {100*t['macro_f1'][0]:.1f}±{100*t['macro_f1'][1]:.1f} MCC {t['mcc'][0]:.3f} BA {100*t['bal_acc'][0]:.1f} P {100*t['macro_precision'][0]:.1f} R {100*t['macro_recall'][0]:.1f} FO {100*t['fallout'][0]:.1f}")
    h = d["tab"]["HWNB"]; print("  HWNB per grade recall", np.round(100*h["recall_g"],1), "f1", np.round(100*h["f1_g"],1), "fallout", np.round(100*h["fallout_g"],1))
    print("  leaky", {k: {m: round(100*v[m],1) if m!='mcc' else round(v[m],3) for m in v} for k, v in d["leaky"].items()}, "genes", d["leaky_genes"][:2])
    print("  top freq", d["freq"].head(8).round(2).to_dict())
    print("  tau", {k: (round(v.statistic,3), round(v.pvalue,4)) for k, v in d["tau"].items()}, "universe", d["agree_universe"], "robust", d["robust"])
print("\nFriedman", fr_stat, "\nranks", dict(zip(CLASSIFIERS, np.round(ranks, 2))), "CD", round(CD, 2))
print("\nRRA top\n", R.head(15))
print("\nABL"); [print(" ", v, {k: (np.round(x, 3) if not isinstance(x, tuple) else tuple(np.round(x, 3))) for k, x in d.items()}) for v, d in ABL.items()]
print("\nLODO"); [print(" ", c, v["n_train"], v["n_test"], v["n_cfs"], v["n_cons"], {k: (round(100*x["macro_f1"],1), round(x["mcc"],3)) for k, x in v["res"].items()}, v["genes"]) for c, v in LODO.items()]